# M0.5 · Tensor araçları ve benchmark disiplini

Bir tensörün bellekte nasıl durduğunu (shape, stride, offset) elle hesaplıyor, stride'larla kopyasız transpoze ve kayan pencere yazıyor, NumPy'da `as_strided` ile aynı numarayı yapıyor, PyTorch'ta view/reshape/contiguous farkını ölçüyor ve en sonda doğru benchmark alışkanlıklarını `torch.utils.benchmark` ile kuruyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m0-5-tensor-tools.html](https://m-gonul.github.io/pytorch-lab/m0-5-tensor-tools.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m0_5_tensor_tools.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def cs():\n        f = need(ns, \"contiguous_strides\")\n        for shape, want in [((2, 3), (3, 1)), ((2, 3, 4), (12, 4, 1)), ((5,), (1,)), ((1, 7, 1, 2), (14, 2, 2, 1))]:\n            ensure(tuple(f(shape)) == want, f\"{shape} → {tuple(f(shape))}, {want} bekleniyordu\")\n    R.test(\"contiguous_strides\", cs)\n\n    data = list(range(24))\n\n    def gt():\n        f = need(ns, \"get\")\n        ensure(f(list(range(6)), (3, 1), 0, (1, 2)) == 5, \"A[1,2] = 5 bekleniyordu\")\n        ensure(f(data, (12, 4, 1), 0, (1, 2, 3)) == 23, \"(2,3,4) tensörde [1,2,3] = 23\")\n        ensure(f(data, (12, 4, 1), 4, (0, 1, 1)) == 9, \"offset hesaba katılmıyor\")\n    R.test(\"get: adres formülü (offset dahil)\", gt)\n\n    def tr():\n        sh, st = need(ns, \"transpose\")((2, 3, 4), (12, 4, 1), 0, 2)\n        ensure(tuple(sh) == (4, 3, 2) and tuple(st) == (1, 4, 12), f\"{sh}, {st}\")\n        m = need(ns, \"materialize\")(data, tuple(sh), tuple(st))\n        want = [[[data[k * 12 + j * 4 + i] for k in range(2)] for j in range(3)] for i in range(4)]\n        ensure(m == want, \"transpoze edilmiş materialize yanlış\")\n    R.test(\"transpose + materialize (3 boyut)\", tr)\n\n    def win():\n        sh, st = need(ns, \"windows\")(6, 3)\n        ensure(need(ns, \"materialize\")(list(range(10, 16)), tuple(sh), tuple(st)) == [[10, 11, 12], [11, 12, 13], [12, 13, 14], [13, 14, 15]], \"pencereler yanlış\")\n        sh, st = need(ns, \"windows\")(5, 5)\n        ensure(tuple(sh) == (1, 5), \"w = n için tek pencere\")\n    R.test(\"windows: kopyasız kayan pencere\", win)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    x = np.random.default_rng(0).normal(size=50)\n\n    def sw():\n        W = need(ns, \"sliding_windows\")(x, 7)\n        ensure(W.shape == (44, 7), f\"shape {W.shape}\")\n        ensure(np.shares_memory(W, x), \"pencereler kopya; x ile bellek paylaşılmalı\")\n        close(W, np.array([x[i:i + 7] for i in range(44)]))\n        return f\"strides (bayt) = {W.strides}\"\n    R.test(\"sliding_windows: doğru ve kopyasız\", sw)\n    R.test(\"moving_average\", lambda: close(need(ns, \"moving_average\")(x, 5), np.convolve(x, np.ones(5) / 5, mode=\"valid\")))\n\n    def iv():\n        f = need(ns, \"is_view_of\")\n        A = np.arange(12.0).reshape(3, 4)\n        ensure(f(A.T, A) and f(A[:, 1], A) and not f(A.copy(), A) and not f(A[[0, 2]], A), \"view/kopya ayrımı yanlış (fancy indexing kopya yapar)\")\n        return \"A.T ve A[:, 1] view; A.copy() ve A[[0, 2]] kopya\"\n    R.test(\"is_view_of: view vs kopya\", iv)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    A = torch.arange(12.0).reshape(3, 4)\n\n    def lay():\n        f = need(ns, \"layout\")\n        ensure(tuple(f(A)) == ((3, 4), (4, 1), 0, True), f\"A: {f(A)}\")\n        ensure(tuple(f(A.T)) == ((4, 3), (1, 4), 0, False), f\"A.T: {f(A.T)}\")\n        ensure(tuple(f(A[1:, ::2])) == ((2, 2), (4, 2), 4, False), f\"A[1:, ::2]: {f(A[1:, ::2])}\")\n        return \"A[1:, ::2] → stride (4, 2), offset 4\"\n    R.test(\"layout: shape, stride, offset, contiguous\", lay)\n\n    def ss():\n        f = need(ns, \"same_storage\")\n        ensure(f(A, A.T) and f(A, A[1:, ::2]) and f(A, A.view(-1)), \"view'lar aynı storage'ı paylaşmalı\")\n        ensure(not f(A, A.clone()) and not f(A, A.T.contiguous()), \"kopyalar farklı storage\")\n    R.test(\"same_storage\", ss)\n\n    def ft():\n        out = need(ns, \"flatten_transposed\")(A)\n        close(out, A.T.contiguous().view(-1))\n        ensure(out.is_contiguous(), \"sonuç contiguous olmalı\")\n    R.test(\"flatten_transposed doğru\", ft)\n\n    def wt():\n        x = torch.arange(20.0)\n        W = need(ns, \"windows_torch\")(x, 4)\n        ensure(tuple(W.shape) == (17, 4), f\"shape {tuple(W.shape)}\")\n        ensure(W.untyped_storage().data_ptr() == x.untyped_storage().data_ptr(), \"pencereler kopya\")\n        close(W, torch.stack([x[i:i + 4] for i in range(17)]))\n        return f\"stride = {W.stride()}\"\n    R.test(\"windows_torch: kopyasız unfold\", wt)\n",
 "e": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    x = torch.randn(5000, generator=g)\n\n    def ma():\n        f = need(ns, \"moving_average_fast\")\n        for w in (1, 3, 64, 5000):\n            close(f(x, w), x.unfold(0, w, 1).double().mean(1), rtol=1e-5, atol=1e-5, what=f\"w = {w}\")\n    R.test(\"moving_average_fast doğru (w = 1, 3, 64, n)\", ma)\n    if \"time_fn\" not in ns or \"moving_average_fast\" not in ns:\n        R.test(\"`time_fn` tanımlı\", lambda: need(ns, \"time_fn\"))\n        return\n    tf, maf = ns[\"time_fn\"], ns[\"moving_average_fast\"]\n\n    def tt():\n        y = torch.randn(200_000)\n        t = tf(torch.sort, y)\n        ensure(isinstance(t, float) and t > 0, f\"pozitif float bekleniyordu: {t!r}\")\n        ts = []\n        for _ in range(7):\n            t0 = time.perf_counter(); torch.sort(y); ts.append(time.perf_counter() - t0)\n        ref = sorted(ts)[3]\n        ensure(0.2 < t / ref < 5, f\"time_fn {fmt_time(t)}, bağımsız ölçüm {fmt_time(ref)} — tutarsız\")\n        return f\"torch.sort(200k): {fmt_time(t)} (bağımsız ölçüm {fmt_time(ref)})\"\n    R.test(\"time_fn tutarlı bir medyan veriyor\", tt)\n\n    y = torch.randn(200_000)\n\n    def loop(v, w):\n        out, s = [], float(v[:w].sum())\n        out.append(s / w)\n        vv = v.tolist()\n        for i in range(w, len(vv)):\n            s += vv[i] - vv[i - w]\n            out.append(s / w)\n        return out\n\n    rows = []\n    for w in (5, 500):\n        k = torch.full((1, 1, w), 1.0 / w)\n        meths = [(\"Python döngüsü (kayan toplam)\", lambda w=w: loop(y, w)),\n                 (\"unfold(…).mean(1)\", lambda w=w: y.unfold(0, w, 1).mean(1)),\n                 (\"cumsum farkı\", lambda w=w: (lambda c: (c[w:] - c[:-w]) / w)(torch.cat([y.new_zeros(1), y.cumsum(0)]))),\n                 (\"F.conv1d\", lambda w=w, k=k: F.conv1d(y.view(1, 1, -1), k).view(-1)),\n                 (\"senin moving_average_fast\", lambda w=w: maf(y, w))]\n        times = [(n, tf(f)) for n, f in meths]\n        best = min(t for _, t in times)\n        rows += [[f\"w = {w}\", n, fmt_time(t), \"en hızlı\" if t == best else f\"{t / best:,.1f}×\"] for n, t in times]\n    R.table(\"Hareketli ortalama, n = 200 000\", [\"pencere\", \"yöntem\", \"medyan süre\", \"en hızlıya göre\"], rows,\n            note=\"unfold O(n·w) iş yapar, cumsum O(n): w büyüdükçe fark açılır. Ölçmeden bilinmez.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m0_5_tensor_tools as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir tensör, düz bir bellek bloğu (**storage**) üzerine konmuş bir bakış açısıdır. Bakışı tanımlayan üç şey:

- **shape** — her eksenin uzunluğu,
- stride — o eksende bir adım ilerlemek için bellekte kaç **eleman** atlanacağı,
- **offset** — ilk elemanın storage içindeki konumu.

$$
\text{adres}(i_0,\dots,i_{k-1})=\text{offset}+\sum_{j}i_j\cdot\text{stride}_j .
$$

Row-major (C sırası) bir $(n, m)$ tensörün stride'ı $(m,1)$'dir. Transpoze, dilim (`x[:, ::2]`), `expand`, `unfold` ve `view` yeni bellek ayırmaz; yalnız shape/stride/offset üçlüsünü değiştirir. Bunlar O(1) işlemlerdir. Buna karşılık contiguous olmayan bir tensör üzerinde çalışan bazı kernel'lar önce kopya ister; gizli kopyalar ileri PyTorch'ta en yaygın performans kaybıdır.

Modülün ikinci yarısı ölçüm: ısınma (ilk çağrıda lazy init, önbellek, `torch.compile`), birden çok tekrar, ortalama değil **medyan**, GPU'da senkronizasyon.

## 1 · Elle

$A=\begin{bmatrix}a_{00}&a_{01}&a_{02}\\a_{10}&a_{11}&a_{12}\end{bmatrix}$ bellekte satır satır yan yana durur. Bir kâğıda 6 kutuluk bir şerit çiz ve indeksleri yaz.

<!-- çözüm -->

Sonra $A^\top$'u düşün: yeni bir şerit çizme; aynı şeridi farklı adımlarla geziyorsun. $B[2,1]$ ile $A[1,2]$'nin aynı kutuya düştüğünü göreceksin. Kayan pencerelerde de aynı numara: her pencere bir öncekinden 1 eleman sonra başlıyor ve pencereler **örtüşüyor**; kopya olsaydı $w$ kat bellek gerekirdi.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$A$: shape $(2,3)$, row-major (C sırası), float32; elemanlar bellekte $[a_{00},a_{01},a_{02},a_{10},a_{11},a_{12}]$**

- `s0`: stride$_0$ (eleman cinsinden)
- `s1`: stride$_1$
- `aoff`: $A[1,2]$'nin offset'i
- `bytes`: $(1000\times1000)$ float32 kaç bayt?

**$B=A^\top$: shape $(3,2)$, aynı bellek, yalnız stride'lar yer değiştirir**

- `t0`: $B$ stride$_0$
- `t1`: $B$ stride$_1$
- `toff`: $B[2,1]$'in offset'i
- `contig`: $B$ contiguous mu? (evet 1, hayır 0)

**Kayan pencere: $x$ uzunluk 6, pencere $w=3$; pencereler bir $(n-w+1)\times w$ view**

- `nwin`: pencere sayısı
- `ws0`: view stride$_0$
- `ws1`: view stride$_1$

In [ ]:
cevaplar = dict(
    s0=None,
    s1=None,
    aoff=None,
    bytes=None,
    t0=None,
    t1=None,
    toff=None,
    contig=None,
    nwin=None,
    ws0=None,
    ws1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"s0":[3,0],"s1":[1,0],"aoff":[5,0],"bytes":[4000000,0],"t0":[1,0],"t1":[3,0],"toff":[5,0],"contig":[0,0],"nwin":[4,0],"ws0":[1,0],"ws1":[1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python'da kendi minik "tensör"ünü yaz: düz bir liste (`data`) + shape + stride + offset.

- `contiguous_strides(shape)` — row-major stride'lar, ör. `(2, 3, 4) → (12, 4, 1)`.
- `get(data, strides, offset, index)` — adres formülüyle tek eleman.
- `transpose(shape, strides, i, j)` → `(yeni_shape, yeni_strides)`; veri kopyalanmaz.
- `materialize(data, shape, strides, offset)` — iç içe liste üret (herhangi bir boyut sayısı; özyineleme ya da döngü).
- `windows(n, w)` → kayan pencere view'u için `(shape, strides)`; `materialize` ile açınca pencereler çıkmalı.

`numpy`/`torch` yasak.

In [ ]:
%%writefile stage_b.py
def contiguous_strides(shape):
    ...


def get(data, strides, offset, index):
    ...


def transpose(shape, strides, i, j):
    ...


def materialize(data, shape, strides, offset=0):
    ...


def windows(n, w):
    """Uzunluk n dizide w'lik kayan pencere view'u → (shape, strides)."""
    ...


data = [0, 1, 2, 3, 4, 5]                      # A = [[0,1,2],[3,4,5]]
st = contiguous_strides((2, 3))
print(st, get(data, st, 0, (1, 2)))             # (3, 1) 5
sh, st2 = transpose((2, 3), st, 0, 1)
print(materialize(data, sh, st2))               # [[0,3],[1,4],[2,5]]
print(materialize(data, *windows(6, 3)))        # [[0,1,2],[1,2,3],[2,3,4],[3,4,5]]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da stride'lar **bayt** cinsindendir (`a.strides`, float64 için eleman başına 8 bayt). `np.lib.stride_tricks.as_strided` istenen shape/stride ile bir view kurar; yanlış stride verilirse bellek dışını okur, bu yüzden güvenli sarmalayıcı `sliding_window_view` tercih edilir.

**Döngü yok:**

- `sliding_windows(x, w)` — `(n-w+1, w)` view; `as_strided` ya da `sliding_window_view` ile. Kontrol, sonucun `x` ile belleği **paylaştığını** (`np.shares_memory`) doğrular.
- `moving_average(x, w)` — pencerelerin ortalaması.
- `is_view_of(a, base)` — `a`, `base`'in belleğini mi gösteriyor? (`np.shares_memory`)

In [ ]:
%%writefile stage_c.py
import numpy as np


def sliding_windows(x, w):
    ...


def moving_average(x, w):
    ...


def is_view_of(a, base):
    ...


x = np.arange(6.0)
W = sliding_windows(x, 3)
print(W, W.strides, is_view_of(W, x))
print(moving_average(x, 3))          # [1. 2. 3. 4.]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta aynı bilgiler: `t.shape`, `t.stride()`, `t.storage_offset()`, `t.is_contiguous()`, `t.untyped_storage().data_ptr()`.

- `layout(t)` → `(tuple(shape), stride(), storage_offset(), is_contiguous())`.
- `same_storage(a, b)` — iki tensör aynı storage'ı mı paylaşıyor?
- `flatten_transposed(x)` — `x.T`'yi düzleştir. `view` kullanırsan hata alırsın; doğru çağrıyı seç.
- `windows_torch(x, w)` — `x.unfold(0, w, 1)`; kopyasız `(n-w+1, w)`.

In [ ]:
%%writefile stage_d.py
import torch


def layout(t):
    ...


def same_storage(a, b):
    ...


def flatten_transposed(x):
    ...


def windows_torch(x, w):
    ...


A = torch.arange(6.0).reshape(2, 3)
print(layout(A), layout(A.T))
print(flatten_transposed(A))       # tensor([0., 3., 1., 4., 2., 5.])
print(windows_torch(torch.arange(6.0), 3))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Hareketli ortalamanın dört yolu var; hangisinin hızlı olduğunu **ölçmeden** söyleyemezsin:

1. Python döngüsü,
2. `x.unfold(0, w, 1).mean(1)` — $O(nw)$ iş, kopyasız view,
3. `cumsum` farkı: $S_i=\sum_{k\le i}x_k$, ortalama $(S_{i+w}-S_i)/w$ — $O(n)$ iş,
4. `F.conv1d` ile $1/w$'lik çekirdek.

- `moving_average_fast(x, w)` — 2, 3 ya da 4; **döngü yok**. (3'te kayan nokta birikimine dikkat: uzun dizilerde `float64` kullan.)
- `time_fn(fn, *args)` → saniye cinsinden medyan süre; `torch.utils.benchmark.Timer(stmt="fn(*args)", globals={...}).blocked_autorange().median`. Bu sınıf ısınmayı, tekrar sayısını ve (GPU'da) senkronizasyonu yönetir.

Kontrol, `time_fn`'in bağımsız bir ölçümle tutarlı olduğunu doğrular ve dört yöntemi iki farklı $w$ için yarıştırır: $w$ büyüdükçe kazanan değişir.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F
from torch.utils import benchmark


def moving_average_fast(x, w):
    """x: (n,) → (n-w+1,). Döngü yok."""
    ...


def time_fn(fn, *args):
    """Medyan süre (saniye), torch.utils.benchmark ile."""
    ...


x = torch.randn(100_000)
print(moving_average_fast(x, 5)[:3])
print(time_fn(moving_average_fast, x, 5))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Harris ve ark. (2020): NumPy ile dizi programlama

[Harris ve ark. — Array programming with NumPy (Nature 2020)](https://arxiv.org/abs/2006.10256)

Makalenin Şekil 1'deki kavramları (şekil, adımlar/strides, görünüm, yayma) sıfırdan küçük bir dizi sınıfıyla yeniden kur: düz bir Python listesi + shape + strides + offset; transpose, reshape (kopyasız mümkünse), dilimleme ve iki dizi arasında yayma kurallarıyla eleman bazlı toplamayı yaz. Sonuçları np.lib.stride_tricks.as_strided ile karşılaştır; hangi reshape'lerin kopya gerektirdiğini strides üzerinden açıkla.


**Adımlar**

1. Yayma: sağdan hizala, boyut 1 olanlar genişler, diğer uyumsuzluk hata.

**Kabul ölçütleri**

- Kendi dizinin sonuçları NumPy ile aynı.
- Kopya gerektiren reshape örneklerinin strides gerekçesi.

## Kaynaklar

- [PyTorch — Tensor Views](https://docs.pytorch.org/docs/stable/tensor_view.html) — BSD-3
- [NumPy — Internal memory layout of an ndarray / as_strided](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.as_strided.html) — BSD-3
- [PyTorch Tutorials — PyTorch Benchmark (torch.utils.benchmark)](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html) — BSD-3
- [Edward Z. Yang — PyTorch internals](http://blog.ezyang.com/2019/05/pytorch-internals/) — ©